In [1]:
import py4j
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
filepath = r"D:\spark_abd\indian_food - indian_food.csv"

In [3]:
spark = SparkSession.builder.master("local[*]").appName("Indian food").getOrCreate()

In [5]:
ind_df = spark.read.csv(filepath,header=True,inferSchema=True)
ind_df.collect()

[Row(name='Balu shahi', ingredients='Maida flour, yogurt, oil, sugar', diet='vegetarian', prep_time=45, cook_time=25, flavor_profile='sweet', course='dessert', state='West Bengal'),
 Row(name='Boondi', ingredients='Gram flour, ghee, sugar', diet='vegetarian', prep_time=80, cook_time=30, flavor_profile='sweet', course='dessert', state='Rajasthan'),
 Row(name='Gajar ka halwa', ingredients='Carrots, milk, sugar, ghee, cashews, raisins', diet='vegetarian', prep_time=15, cook_time=60, flavor_profile='sweet', course='dessert', state='Punjab'),
 Row(name='Ghevar', ingredients='Flour, ghee, kewra, milk, clarified butter, sugar, almonds, pistachio, saffron, green cardamom', diet='vegetarian', prep_time=15, cook_time=30, flavor_profile='sweet', course='dessert', state='Rajasthan'),
 Row(name='Gulab jamun', ingredients='Milk powder, plain flour, baking powder, ghee, milk, sugar, water, rose water', diet='vegetarian', prep_time=15, cook_time=40, flavor_profile='sweet', course='dessert', state='Wes

In [6]:
ind_df.printSchema()

root
 |-- name: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- diet: string (nullable = true)
 |-- prep_time: integer (nullable = true)
 |-- cook_time: integer (nullable = true)
 |-- flavor_profile: string (nullable = true)
 |-- course: string (nullable = true)
 |-- state: string (nullable = true)



In [7]:
ind_df.show()

+--------------+--------------------+----------+---------+---------+--------------+-------+-------------+
|          name|         ingredients|      diet|prep_time|cook_time|flavor_profile| course|        state|
+--------------+--------------------+----------+---------+---------+--------------+-------+-------------+
|    Balu shahi|Maida flour, yogu...|vegetarian|       45|       25|         sweet|dessert|  West Bengal|
|        Boondi|Gram flour, ghee,...|vegetarian|       80|       30|         sweet|dessert|    Rajasthan|
|Gajar ka halwa|Carrots, milk, su...|vegetarian|       15|       60|         sweet|dessert|       Punjab|
|        Ghevar|Flour, ghee, kewr...|vegetarian|       15|       30|         sweet|dessert|    Rajasthan|
|   Gulab jamun|Milk powder, plai...|vegetarian|       15|       40|         sweet|dessert|  West Bengal|
|        Imarti|Sugar syrup, lent...|vegetarian|       10|       50|         sweet|dessert|  West Bengal|
|        Jalebi|Maida, corn flour...|vegetaria

1. Find out how many unique dishes are present

In [8]:
ind_df.createOrReplaceTempView('Indian')


In [13]:
spark.sql('Select count(distinct(name)) from Indian').show()

+--------------------+
|count(DISTINCT name)|
+--------------------+
|                 255|
+--------------------+



2. Which state has more dishes?

In [36]:
spark.sql('Select count(distinct(name)) as dishcount,state from Indian group by state order by dishcount desc limit 1').show()

+---------+-------+
|dishcount|  state|
+---------+-------+
|       35|Gujarat|
+---------+-------+



Using window function

In [35]:
spark.sql('WITH RankedStates AS (SELECT state,COUNT(DISTINCT name) AS dishcount,DENSE_RANK() OVER (ORDER BY COUNT(DISTINCT name) DESC) AS rnk FROM Indian GROUP BY state) SELECT dishcount, state FROM RankedStates WHERE rnk = 1').show()

+---------+-------+
|dishcount|  state|
+---------+-------+
|       35|Gujarat|
+---------+-------+



3. How many dishes from state Karnataka?

In [39]:
spark.sql('Select count(distinct(name)) from Indian where state = "Karnataka"').show()

+--------------------+
|count(DISTINCT name)|
+--------------------+
|                   6|
+--------------------+



4. List number of unique regions

In [49]:
spark.sql('Select count(distinct(state)) as region_count from Indian').show()

+------------+
|region_count|
+------------+
|          25|
+------------+



5. Count number of dishes from each region.

In [70]:
spark.sql('Select count(distinct(name)),state from Indian group by state').show()

+--------------------+---------------+
|count(DISTINCT name)|          state|
+--------------------+---------------+
|                   1|       Nagaland|
|                   6|      Karnataka|
|                  24|             -1|
|                   7|         Odisha|
|                   8|         Kerala|
|                  20|     Tamil Nadu|
|                   1|   Chhattisgarh|
|                  10| Andhra Pradesh|
|                   2| Madhya Pradesh|
|                  32|         Punjab|
|                   2|        Manipur|
|                   2|Jammu & Kashmir|
|                   3|            Goa|
|                   1|        Haryana|
|                  35|        Gujarat|
|                   6|      Rajasthan|
|                  21|          Assam|
|                   1|   NCT of Delhi|
|                  24|    West Bengal|
|                  30|    Maharashtra|
+--------------------+---------------+
only showing top 20 rows



6. List unique 'flavor_profile' and 'course'

In [58]:
spark.sql('Select distinct(flavor_profile,course) from Indian').show()

+------------------------------------------------------------+
|named_struct(flavor_profile, flavor_profile, course, course)|
+------------------------------------------------------------+
|                                             {bitter, snack}|
|                                            {spicy, starter}|
|                                         {sour, main course}|
|                                                 {-1, snack}|
|                                           {-1, main course}|
|                                        {sweet, main course}|
|                                        {bitter, main cou...|
|                                              {spicy, snack}|
|                                            {sweet, dessert}|
|                                        {spicy, main course}|
+------------------------------------------------------------+



In [60]:
ind_df.show()

+--------------+--------------------+----------+---------+---------+--------------+-------+-------------+
|          name|         ingredients|      diet|prep_time|cook_time|flavor_profile| course|        state|
+--------------+--------------------+----------+---------+---------+--------------+-------+-------------+
|    Balu shahi|Maida flour, yogu...|vegetarian|       45|       25|         sweet|dessert|  West Bengal|
|        Boondi|Gram flour, ghee,...|vegetarian|       80|       30|         sweet|dessert|    Rajasthan|
|Gajar ka halwa|Carrots, milk, su...|vegetarian|       15|       60|         sweet|dessert|       Punjab|
|        Ghevar|Flour, ghee, kewr...|vegetarian|       15|       30|         sweet|dessert|    Rajasthan|
|   Gulab jamun|Milk powder, plai...|vegetarian|       15|       40|         sweet|dessert|  West Bengal|
|        Imarti|Sugar syrup, lent...|vegetarian|       10|       50|         sweet|dessert|  West Bengal|
|        Jalebi|Maida, corn flour...|vegetaria

7. Which state has more 'main course'?

In [65]:
spark.sql('select state,count(course) main_course from Indian group by state order by main_course desc limit 1').show()

+-------+-----------+
|  state|main_course|
+-------+-----------+
|Gujarat|         35|
+-------+-----------+



8. Give the %of dishes from each region.

In [76]:
spark.sql('select round((count(name)/236)*100,2) as perc ,state from Indian group by state').show()

+-----+---------------+
| perc|          state|
+-----+---------------+
| 0.42|       Nagaland|
| 2.54|      Karnataka|
|10.17|             -1|
| 2.97|         Odisha|
| 3.39|         Kerala|
| 8.47|     Tamil Nadu|
| 0.42|   Chhattisgarh|
| 4.24| Andhra Pradesh|
| 0.85| Madhya Pradesh|
|13.56|         Punjab|
| 0.85|        Manipur|
| 0.85|Jammu & Kashmir|
| 1.27|            Goa|
| 0.42|        Haryana|
|14.83|        Gujarat|
| 2.54|      Rajasthan|
|  8.9|          Assam|
| 0.42|   NCT of Delhi|
|10.17|    West Bengal|
|12.71|    Maharashtra|
+-----+---------------+
only showing top 20 rows



9. List the states which has more dishes from each region.

In [78]:
spark.sql('Select count(distinct(name)) dishcount,state from Indian group by state order by dishcount desc limit 10').show()

+---------+--------------+
|dishcount|         state|
+---------+--------------+
|       35|       Gujarat|
|       32|        Punjab|
|       30|   Maharashtra|
|       24|            -1|
|       24|   West Bengal|
|       21|         Assam|
|       20|    Tamil Nadu|
|       10|Andhra Pradesh|
|        9| Uttar Pradesh|
|        8|        Kerala|
+---------+--------------+

